# Damaten HexAI vs MoHex benchmark

Runs on Colab (Linux). Execute the cells top to bottom:
1. build MoHex (benzene)
2. clone Damaten + build HexAI (no LFS needed) + sanity-check HTP
3. run the benchmark (shakedown, then sweep MoHex's time budget to the ~50% point)

If a cell errors (MoHex build or board convention), paste the log.

In [ ]:
# 1) Build MoHex (benzene). Takes a few minutes.
!apt-get -qq install -y libboost-all-dev libdb-dev cmake >/dev/null
![ -d benzene-vanilla-cmake ] || git clone -q https://github.com/cgao3/benzene-vanilla-cmake.git
!cd benzene-vanilla-cmake && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j4
!ls -la benzene-vanilla-cmake/build/src/mohex/mohex

In [ ]:
# 2) Clone Damaten (model is a normal git file now -- no LFS), build HexAI, check HTP
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://github.com/Koushien552/Damaten.git || (cd Damaten && git pull -q --ff-only)
!g++ -O3 -std=c++17 -mavx2 -mfma -o hexai Damaten/src/main.cpp
!echo "model:" $(head -c 30 Damaten/models/hex_model.nn)
!printf 'name\nquit\n' | ./hexai htp

In [ ]:
# 3) Run the benchmark
M = "benzene-vanilla-cmake/build/src/mohex/mohex"
# shakedown: auto-detect board convention + one game
!python Damaten/colab/bench_vs_mohex.py --mohex {M} --hexai ./hexai --model Damaten/models/hex_model.nn --verify-only
# full: HexAI fixed at 1500 iters; sweep MoHex time/move; 100 games at the ~50% point
!python Damaten/colab/bench_vs_mohex.py --mohex {M} --hexai ./hexai --model Damaten/models/hex_model.nn \
  --hexai-iters 1500 --budgets 0.1,0.5,2.0 --coarse-games 20 --final-games 100 --out bench_results.csv